# AI Game Assistant — Complete Project

**Stack:** Python + Pandas + scikit-learn retrieval + LangChain prompt orchestration + Streamlit UI.

This notebook is designed to be stable on modern Python (including Python 3.13). It deliberately avoids the `transformers / sentence-transformers / PEFT / FAISS` dependency chain that caused the earlier import errors.

It:
- reuses an existing local Steam dataset if available;
- otherwise tries the Hugging Face Dataset Viewer REST API;
- falls back to bundled demo data instead of crashing if the network is unavailable;
- cleans and prepares the game data;
- builds a local TF-IDF retrieval engine;
- understands natural-language price/platform filters;
- optionally uses LangChain + Groq for conversational LLM answers;
- generates a complete Streamlit `app.py`;
- writes `requirements.txt`, `README.md`, and `.gitignore`;
- runs smoke tests at the end.

The assistant still works without an API key; the LLM layer is optional.

## 1. Environment and dependency check

In [1]:
import sys, subprocess, importlib.util
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)

CORE = {
    "pandas": "pandas>=2.2",
    "numpy": "numpy>=1.26",
    "sklearn": "scikit-learn>=1.5",
    "requests": "requests>=2.31",
    "joblib": "joblib>=1.3",
}
OPTIONAL = {
    "streamlit": "streamlit>=1.35",
    "langchain_core": "langchain-core>=0.3",
}

def ensure_packages(package_map, required=True):
    missing = [pip_name for module, pip_name in package_map.items()
               if importlib.util.find_spec(module) is None]
    if not missing:
        return True
    print("Missing packages:", ", ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print("Installation completed.")
        return True
    print("Automatic installation could not complete in this environment.")
    if required:
        print("Run this once, then restart the kernel:")
        print(" ".join(cmd))
    else:
        print("The notebook can still run; these are only needed to launch app.py.")
    return False

core_ok = ensure_packages(CORE, required=True)
optional_ok = ensure_packages(OPTIONAL, required=False)
print("Core:", "OK" if core_ok else "installation may be required")
print("Streamlit/LangChain:", "OK" if optional_ok else "optional packages not installed here")

Python: 3.13.3
Executable: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3
Core: OK
Streamlit/LangChain: OK


## 2. Imports and folders

In [2]:
import os, re, ast, html, warnings, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)
print("Project:", PROJECT_ROOT)

Project: /Users/adityaraj/Desktop/AI


## 3. Load Steam games data safely

In [3]:
DATASET_NAME = "FronkonGames/steam-games-dataset"
TARGET_ROWS = 2000

LOCAL_CANDIDATES = [
    DATA_DIR / "steam_games_clean.csv",
    DATA_DIR / "steam_games_clean.parquet",
    PROJECT_ROOT / "steam_games_clean.csv",
    PROJECT_ROOT / "steam_games_clean.parquet",
    PROJECT_ROOT.parent / "data" / "steam_games_clean.csv",
    PROJECT_ROOT.parent / "data" / "steam_games_clean.parquet",
]

def load_local_dataset():
    for path in LOCAL_CANDIDATES:
        if not path.exists():
            continue
        try:
            frame = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_parquet(path)
            if len(frame):
                print(f"Loaded local dataset: {path} ({len(frame):,} rows)")
                return frame, f"local:{path}"
        except Exception as exc:
            print(f"Skipped {path.name}: {type(exc).__name__}: {exc}")
    return None, None

def fetch_hf_rows(target_rows=TARGET_ROWS):
    endpoint = "https://datasets-server.huggingface.co/rows"
    rows = []
    for offset in range(0, target_rows, 100):
        params = {
            "dataset": DATASET_NAME, "config": "default", "split": "train",
            "offset": offset, "length": min(100, target_rows-offset),
        }
        r = requests.get(endpoint, params=params, timeout=20)
        r.raise_for_status()
        payload = r.json()
        batch = [item.get("row", {}) for item in payload.get("rows", [])]
        if not batch:
            break
        rows.extend(batch)
    if not rows:
        raise RuntimeError("Dataset API returned no rows.")
    return pd.DataFrame(rows[:target_rows])

def bundled_demo_data():
    games = [
        ("Shadowforge Realms","Action, RPG","Dark Fantasy, Difficult, Souls-like","Single-player","Explore a ruined fantasy kingdom and master precise melee combat.",29.99,1,0,0,9200,800),
        ("Neon Velocity","Racing","Cyberpunk, Fast-Paced, Arcade","Single-player, Multiplayer","High-speed futuristic street racing through a neon megacity.",14.99,1,1,0,5100,900),
        ("Orbital Colony","Strategy, Simulation","Space, Colony Sim, Management","Single-player","Build and manage a fragile human colony on a distant moon.",24.99,1,1,1,6800,700),
        ("Echoes Below","Horror, Adventure","Psychological Horror, Mystery, Atmospheric","Single-player","Investigate an abandoned research facility beneath the ocean.",12.99,1,0,0,4300,600),
        ("Raid Party","Action, Adventure","Co-op, Multiplayer, Fantasy","Co-op, Online Multiplayer","Team up with friends to clear dungeons and defeat giant bosses.",19.99,1,0,0,7600,1200),
        ("Pixel Farm Days","Simulation, Casual","Farming, Relaxing, Cozy","Single-player","Grow crops, meet villagers and restore a peaceful countryside farm.",9.99,1,1,1,8400,500),
        ("Tactical Horizon","Strategy","Turn-Based, Tactical, Sci-Fi","Single-player","Command a squad through tactical missions on a hostile alien world.",34.99,1,0,0,4700,500),
        ("Dungeon Cartographer","RPG, Adventure","Dungeon Crawler, Exploration, Fantasy","Single-player","Map a shifting underground labyrinth filled with secrets and monsters.",17.49,1,1,1,3900,450),
        ("Night Shift Protocol","Horror","Co-op, Survival Horror, Multiplayer","Co-op, Online Multiplayer","Survive a haunted hospital together while completing dangerous objectives.",8.99,1,0,0,7100,1300),
        ("Kingdom Ledger","Strategy, Simulation","Economy, Medieval, Management","Single-player","Run the economy of a medieval kingdom while balancing nobles and citizens.",21.99,1,1,0,3600,350),
        ("Skybound Isles","Adventure","Open World, Exploration, Fantasy","Single-player","Sail between floating islands and uncover the history of a lost civilisation.",27.99,1,1,0,6200,700),
        ("Circuit Arena","Action","Arena Shooter, Multiplayer, Competitive","Online Multiplayer","Competitive arena battles built around movement, gadgets and team tactics.",0.0,1,0,0,10500,2500),
        ("Mech Frontier","Action, Simulation","Mechs, Sci-Fi, Combat","Single-player, Multiplayer","Pilot customisable combat mechs across a war-torn frontier.",39.99,1,0,0,5800,1000),
        ("Quiet Woods","Adventure, Casual","Cozy, Exploration, Nature","Single-player","A calm hiking adventure about photography, wildlife and discovery.",6.99,1,1,1,3100,180),
        ("Chronicle of Ash","RPG","Story Rich, Choices Matter, Fantasy","Single-player","A narrative role-playing game where choices reshape the kingdom.",31.99,1,0,0,6700,650),
        ("Deep Signal","Horror, Sci-Fi","Space, Survival, Atmospheric","Single-player","Repair a silent space station while something stalks the corridors.",18.99,1,0,0,4900,550),
        ("Tabletop Empire","Strategy","Board Game, Turn-Based, Multiplayer","Single-player, Multiplayer","Build an empire in a digital strategy board game with online matches.",11.99,1,1,1,2800,300),
        ("Velocity Football","Sports","Football, Multiplayer, Competitive","Single-player, Multiplayer","Fast arcade football with short matches and online leagues.",7.99,1,0,0,4100,900),
        ("Mystic Academy","RPG, Simulation","Magic, School, Life Sim","Single-player","Learn spells, build friendships and explore a magical academy.",22.49,1,1,0,7300,600),
        ("Last Train North","Adventure","Post-Apocalyptic, Story Rich, Survival","Single-player","Lead survivors across a frozen continent aboard an ageing train.",16.99,1,0,0,5400,500),
        ("Cozy Cafe","Simulation, Casual","Cooking, Cozy, Management","Single-player","Design a cafe, create recipes and build relationships with regular customers.",5.99,1,1,1,6000,250),
        ("Arena Legends","Action","MOBA, Multiplayer, Team-Based","Online Multiplayer","Choose a hero and compete in objective-based five-versus-five battles.",0.0,1,1,0,13000,3200),
        ("Forgotten Circuit","Puzzle, Adventure","Puzzle, Mystery, Sci-Fi","Single-player","Solve interconnected machine puzzles inside a deserted automated city.",13.49,1,1,1,3500,250),
        ("Iron Battalion","Strategy, Action","RTS, Military, Multiplayer","Single-player, Multiplayer","Control combined-arms forces in large real-time tactical battles.",28.99,1,0,0,4600,750),
    ]
    rows=[]
    for i,g in enumerate(games,1):
        name,genres,tags,categories,desc,price,win,mac,linux,pos,neg = g
        rows.append({
            "appID": f"demo-{i}", "name": name, "release_date":"2026-01-01",
            "price":price, "detailed_description":desc, "short_description":desc,
            "supported_languages":["English"], "header_image":"",
            "windows":bool(win), "mac":bool(mac), "linux":bool(linux),
            "metacritic_score":0, "positive":pos, "negative":neg,
            "recommendations":pos, "average_playtime_forever":0,
            "developers":["Demo Studio"], "publishers":["Demo Publisher"],
            "categories":categories.split(", "), "genres":genres.split(", "), "tags":tags.split(", "),
        })
    return pd.DataFrame(rows)

raw_df, data_source = load_local_dataset()
if raw_df is None:
    try:
        print(f"No local data found. Trying Hugging Face API for {TARGET_ROWS:,} rows...")
        raw_df = fetch_hf_rows()
        data_source = f"huggingface-api:{len(raw_df)}"
    except Exception as exc:
        print(f"Online fetch unavailable: {type(exc).__name__}: {exc}")
        print("Using bundled demo data so the project still runs end-to-end.")
        raw_df = bundled_demo_data()
        data_source = "bundled-demo"

print("Source:", data_source)
print("Raw shape:", raw_df.shape)
raw_df.head(3)

Loaded local dataset: /Users/adityaraj/Desktop/AI/data/steam_games_clean.csv (20,000 rows)
Source: local:/Users/adityaraj/Desktop/AI/data/steam_games_clean.csv
Raw shape: (20000, 24)


,appID,name,release_date,price,detailed_description,short_description,supported_languages,header_image,windows,mac,linux,metacritic_score,positive,negative,recommendations,average_playtime_forever,developers,publishers,categories,genres,tags,total_reviews,positive_ratio,platforms
0,1282110,Escape Initiative,"Jun 12, 2020",0.00,Escape Initiative is an action precision platf...,"Run, jump, and shoot lasers in this difficult ...",English,https://shared.akamai.steamstatic.com/store_it...,True,False,False,0,17,2,0,0,Jim Bame,Jim Bame,"Single-player, Steam Achievements","Action, Indie",NaN,19,0.894737,Windows
1,1862790,Siienaa,"Mar 18, 2022",2.49,Siienaa is an immersive VR experience that all...,Siienaa is an immersive VR experience that all...,English,https://shared.akamai.steamstatic.com/store_it...,True,False,False,0,10,1,0,0,Snefru,Snefru,"Single-player, Tracked Controller Support, VR ...","Adventure, Casual, Indie, Simulation",NaN,11,0.909091,Windows
2,1757360,Shifting Dungeon,"Nov 10, 2021",2.99,"Solve interesting puzzles, manipulate Shifting...","Solve interesting puzzles, manipulate shifting...",English,https://shared.akamai.steamstatic.com/store_it...,True,False,False,0,2,0,0,0,Vesite,Vesite,"Single-player, Family Sharing","Adventure, Strategy",NaN,2,1.000000,Windows


## 4. Clean and standardise

In [4]:
DEFAULTS = {
    "appID":"", "name":"", "release_date":"", "price":0.0,
    "detailed_description":"", "short_description":"", "supported_languages":[],
    "header_image":"", "windows":False, "mac":False, "linux":False,
    "metacritic_score":0, "positive":0, "negative":0, "recommendations":0,
    "average_playtime_forever":0, "developers":[], "publishers":[],
    "categories":[], "genres":[], "tags":[],
}
for col, default in DEFAULTS.items():
    if col not in raw_df.columns:
        raw_df[col] = [default] * len(raw_df) if isinstance(default, list) else default

def listish_to_text(value):
    if value is None:
        return ""
    if isinstance(value,(list,tuple,set,np.ndarray)):
        return ", ".join(str(x) for x in value if x is not None)
    if isinstance(value,float) and np.isnan(value):
        return ""
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed,(list,tuple,set)):
                return ", ".join(str(x) for x in parsed if x is not None)
        except Exception:
            pass
    return text

def clean_html_text(value):
    text = listish_to_text(value)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html.unescape(text)
    return re.sub(r"\s+", " ", text).strip()

df = raw_df[list(DEFAULTS)].copy()
for col in ["name","short_description","detailed_description","header_image","release_date"]:
    df[col] = df[col].map(clean_html_text)
for col in ["supported_languages","developers","publishers","categories","genres","tags"]:
    df[col] = df[col].map(listish_to_text)
for col in ["price","metacritic_score","positive","negative","recommendations","average_playtime_forever"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
for col in ["windows","mac","linux"]:
    # Robust conversion for bools, 0/1, or strings.
    df[col] = df[col].map(lambda x: str(x).strip().lower() in {"true","1","yes"} if not isinstance(x,(bool,np.bool_)) else bool(x))

df = df[df["name"].str.len()>0].copy()
df["short_description"] = df["short_description"].where(df["short_description"].str.len()>0, df["detailed_description"])
df = df[df["short_description"].str.len()>0].drop_duplicates(subset=["appID","name"]).reset_index(drop=True)
df["total_reviews"] = df["positive"] + df["negative"]
df["positive_ratio"] = np.where(df["total_reviews"]>0, df["positive"]/df["total_reviews"], 0.0)

def platforms_for(row):
    out=[]
    if row["windows"]: out.append("Windows")
    if row["mac"]: out.append("Mac")
    if row["linux"]: out.append("Linux")
    return ", ".join(out)

df["platforms"] = df.apply(platforms_for, axis=1)
print("Clean shape:", df.shape)
df[["name","genres","price","platforms","positive_ratio"]].head()

Clean shape: (20000, 24)


,name,genres,price,platforms,positive_ratio
0,Escape Initiative,"Action, Indie",0.00,Windows,0.894737
1,Siienaa,"Adventure, Casual, Indie, Simulation",2.49,Windows,0.909091
2,Shifting Dungeon,"Adventure, Strategy",2.99,Windows,1.000000
3,Parkour Is White?,"Casual, Indie, Racing",1.99,Windows,0.750000
4,Slinki,"Adventure, Indie",0.99,"Windows, Linux",0.583333


## 5. Save clean data

In [5]:
CLEAN_DATA_PATH = DATA_DIR / "steam_games_clean.csv"
df.to_csv(CLEAN_DATA_PATH, index=False)
print("Saved:", CLEAN_DATA_PATH)

Saved: /Users/adityaraj/Desktop/AI/data/steam_games_clean.csv


## 6. Build search text

In [6]:
def make_search_text(row):
    return " ".join([
        str(row["name"]),
        f'genres {row["genres"]}',
        f'categories {row["categories"]}',
        f'tags {row["tags"]}',
        f'description {row["short_description"]}',
        f'details {row["detailed_description"]}',
        f'developer {row["developers"]}',
        f'publisher {row["publishers"]}',
        f'platforms {row["platforms"]}',
    ])

df["search_text"] = df.apply(make_search_text, axis=1)
print(df.loc[0,"search_text"][:600])

Escape Initiative genres Action, Indie categories Single-player, Steam Achievements tags  description Run, jump, and shoot lasers in this difficult and aesthetic action precision platformer. Built for fast play but not required, you will enjoy hand built levels with an immense variety of challenges. Customize your colors and realize your initiative. ESCAPE. details Escape Initiative is an action precision platformer where you play as a cyborg awakened in a strange place with an extreme urge to escape. Features Customizable character color and laser color Normal difficulty with forgiving checkp


## 7. Train local retrieval engine

In [7]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_features=25000,
    min_df=1,
    sublinear_tf=True,
    strip_accents="unicode",
)
tfidf_matrix = vectorizer.fit_transform(df["search_text"])
VECTORIZER_PATH = ARTIFACT_DIR / "tfidf_vectorizer.joblib"
MATRIX_PATH = ARTIFACT_DIR / "tfidf_matrix.joblib"
joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(tfidf_matrix, MATRIX_PATH)
print("TF-IDF matrix:", tfidf_matrix.shape)
print("Vocabulary:", len(vectorizer.vocabulary_))

TF-IDF matrix: (20000, 25000)
Vocabulary: 25000


## 8. Search + natural-language filters

In [8]:
PRICE_PATTERN = re.compile(r"(?:under|below|less than|max(?:imum)?|up to)\s*[£$€]?\s*(\d+(?:\.\d+)?)", re.I)

def infer_query_filters(query):
    q=query.lower()
    m=PRICE_PATTERN.search(query)
    max_price=float(m.group(1)) if m else (0.0 if "free" in q else None)
    platform=None
    if re.search(r"\bmac(?:os)?\b", q): platform="mac"
    elif re.search(r"\blinux\b", q): platform="linux"
    elif re.search(r"\bwindows\b|\bpc\b", q): platform="windows"
    return max_price, platform

def search_games(query, top_k=5, max_price=None, platform=None, min_reviews=0):
    if not isinstance(query,str) or not query.strip():
        return pd.DataFrame()
    inferred_price, inferred_platform = infer_query_filters(query)
    if max_price is None: max_price = inferred_price
    if platform is None or str(platform).lower()=="any": platform = inferred_platform

    scores = cosine_similarity(vectorizer.transform([query]), tfidf_matrix).ravel()
    mask=np.ones(len(df),dtype=bool)
    if max_price is not None: mask &= df["price"].to_numpy(dtype=float) <= float(max_price)
    if platform and str(platform).lower() in {"windows","mac","linux"}:
        mask &= df[str(platform).lower()].to_numpy(dtype=bool)
    if min_reviews: mask &= df["total_reviews"].to_numpy(dtype=float) >= int(min_reviews)

    valid=np.flatnonzero(mask)
    if not len(valid): return pd.DataFrame()
    ranked = valid[np.argsort(scores[valid])[::-1]][:top_k]
    cols=["appID","name","genres","tags","short_description","price","platforms",
          "positive_ratio","total_reviews","metacritic_score","header_image"]
    result=df.iloc[ranked][cols].copy()
    result.insert(0,"match_score",scores[ranked])
    return result.reset_index(drop=True)

## 9. Test the recommender

In [9]:
for q in [
    "dark fantasy difficult RPG for Windows",
    "co-op horror game under £20",
    "relaxing cozy simulation on Mac",
]:
    print("\nQUERY:", q)
    result=search_games(q, top_k=3)
    if result.empty:
        print("No games matched.")
    else:
        display(result[["name","genres","price","platforms","positive_ratio","match_score"]])


QUERY: dark fantasy difficult RPG for Windows


,name,genres,price,platforms,positive_ratio,match_score
0,人狼の仔,"Indie, RPG, Early Access",3.99,Windows,0.0,0.343349
1,Camelot 10000CE,"Action, Adventure, Indie",7.19,"Windows, Mac, Linux",0.0,0.242171
2,Dawn of Corruption,"Adventure, Indie, RPG, Early Access",0.00,Windows,0.0,0.240491



QUERY: co-op horror game under £20


,name,genres,price,platforms,positive_ratio,match_score
0,Parapsychology,"Action, Adventure, Casual, Indie, Early Access",3.99,Windows,0.529412,0.336267
1,Astrumis - Survive Together,"Action, Adventure, Indie",6.49,Windows,0.759494,0.227497
2,School Labyrinth,"Action, Indie",5.59,Windows,0.832138,0.226169



QUERY: relaxing cozy simulation on Mac


,name,genres,price,platforms,positive_ratio,match_score
0,Cozy Fishing,"Casual, Indie",1.39,"Windows, Mac, Linux",0.692308,0.208609
1,Eyes That Hypnotise,"Casual, Indie",1.49,"Windows, Mac",1.000000,0.188877
2,Sweet Lyoria: Cozy Homes,"Casual, Indie, Simulation, Early Access",2.24,"Windows, Mac",0.000000,0.169256


## 10. Retrieval-only assistant

In [10]:
def format_price(value):
    value=float(value)
    return "Free" if value<=0 else f"£{value:.2f}"

def fallback_answer(question, results):
    if results is None or results.empty:
        return "I could not find a game matching those filters. Try widening the requirements."
    lines=[f"I found {len(results)} strong match(es) for: **{question}**",""]
    for i,row in results.iterrows():
        reviews=f'{row["positive_ratio"]*100:.0f}% positive' if row["total_reviews"]>0 else "review score unavailable"
        lines.append(
            f'{i+1}. **{row["name"]}** — {row["genres"] or "Genre not listed"} | '
            f'{format_price(row["price"])} | {row["platforms"] or "Platform not listed"} | {reviews}. '
            f'{row["short_description"]}'
        )
    return "\n\n".join(lines)

answer, demo_results = fallback_answer("I want a co-op horror game under £20", search_games("co-op horror under £20",3)), search_games("co-op horror under £20",3)
print(answer)

I found 3 strong match(es) for: **I want a co-op horror game under £20**



1. **Parapsychology** — Action, Adventure, Casual, Indie, Early Access | £3.99 | Windows | 53% positive. Parapsychology is a co-op horror game that can be played with 1-3 people. Find the demon in the house with your friends and destroy it with instructions.

2. **Astrumis - Survive Together** — Action, Adventure, Indie | £6.49 | Windows | 76% positive. Astrumis - Survive Together is an exclusively co-op horror game. One player controls a survivor on an abandoned spaceship and the other the ship itself. They must work together to survive against the mysterious monster.

3. **School Labyrinth** — Action, Indie | £5.59 | Windows | 83% positive. School labyrinth is a 1~4 co-op horror game set in a randomly generated school at night. Cooperate with your friends and try to escape from the school building!


## 11. Optional LangChain + Groq LLM layer

When a Groq API key is provided, this uses a LangChain `ChatPromptTemplate` and `RunnableLambda` to turn retrieved game records into a conversational answer. Without a key, the local retrieval response is used.

In [11]:
def langchain_available():
    return importlib.util.find_spec("langchain_core") is not None

def resolve_groq_model(api_key, preferred=None):
    headers={"Authorization":f"Bearer {api_key}"}
    r=requests.get("https://api.groq.com/openai/v1/models",headers=headers,timeout=20)
    r.raise_for_status()
    ids=[x.get("id","") for x in r.json().get("data",[])]
    if preferred and preferred in ids: return preferred
    for keyword in ["llama","qwen","gemma","mixtral"]:
        found=next((x for x in ids if keyword in x.lower()),None)
        if found: return found
    if not ids: raise RuntimeError("No Groq models returned.")
    return ids[0]

def results_context(results):
    parts=[]
    for _,row in results.iterrows():
        parts.append(
            f'Game: {row["name"]}\nGenres: {row["genres"]}\n'
            f'Price: {format_price(row["price"])}\nPlatforms: {row["platforms"]}\n'
            f'Positive ratio: {row["positive_ratio"]:.3f}\nReviews: {int(row["total_reviews"])}\n'
            f'Description: {row["short_description"]}'
        )
    return "\n\n---\n\n".join(parts)

def answer_with_optional_llm(question, top_k=5, api_key=None, model=None, max_price=None, platform=None):
    results=search_games(question,top_k,max_price,platform)
    if results.empty:
        return fallback_answer(question,results),results,"retrieval-only"
    api_key=(api_key or os.getenv("GROQ_API_KEY","")).strip()
    if not api_key or not langchain_available():
        return fallback_answer(question,results),results,"retrieval-only"
    try:
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.runnables import RunnableLambda
        chosen_model=resolve_groq_model(api_key,model)
        prompt=ChatPromptTemplate.from_messages([
            ("system","You are an AI Game Assistant. Use only the retrieved game context. "
                      "Recommend suitable games and explain why they match. Never invent facts."),
            ("human","Request:\n{question}\n\nRetrieved context:\n{context}")
        ])
        def call_groq(prompt_value):
            messages=[]
            for msg in prompt_value.to_messages():
                role="assistant" if msg.type=="ai" else ("system" if msg.type=="system" else "user")
                messages.append({"role":role,"content":str(msg.content)})
            payload={"model":chosen_model,"messages":messages,"temperature":0.2}
            headers={"Authorization":f"Bearer {api_key}","Content-Type":"application/json"}
            r=requests.post("https://api.groq.com/openai/v1/chat/completions",headers=headers,json=payload,timeout=45)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        chain=prompt | RunnableLambda(call_groq)
        text=chain.invoke({"question":question,"context":results_context(results)})
        return text,results,f"langchain+groq:{chosen_model}"
    except Exception as exc:
        return fallback_answer(question,results)+f"\n\n[LLM unavailable: {type(exc).__name__}]",results,"retrieval-fallback"

print("LangChain installed:", langchain_available())

LangChain installed: True


## 12. Smoke-test assistant

In [12]:
assistant_text, assistant_results, assistant_mode = answer_with_optional_llm(
    "Suggest a multiplayer or co-op game under £20", top_k=4, api_key=None
)
assert isinstance(assistant_text,str) and len(assistant_text)>20
assert isinstance(assistant_results,pd.DataFrame)
print("Mode:", assistant_mode)
print(assistant_text[:1200])

Mode: retrieval-only
I found 4 strong match(es) for: **Suggest a multiplayer or co-op game under £20**



1. **Sagittarius** — Action, Casual, Indie | £7.99 | Windows | 100% positive. Sagittarius is a classic side scrolling space shoot'em up in a modern take. In this epic space game it's up to you to save our solar system from destruction. Fly alone or with a friend in a local multiplayer co-op game.

2. **Cooperate with Yourself** — Action, Free To Play | Free | Windows | review score unavailable. This game is a 2D platformer where a single player controls up to six characters. In other words, it's a game that lets you experience a multiplayer co-op game all by yourself. Work together, use each other as stepping stones when needed, and clear all 16 levels!

3. **Ballfrog** — Indie | £1.49 | Windows | 79% positive. BallFrog is a challenging ball rolling platformer with experimental controls! Use your Tongue to grapple and swing.

4. **Pets at Work** — Adventure, Casual, Indie, Strategy

## 13. Generate Streamlit app

In [13]:
APP_CODE = 'from pathlib import Path\nimport os, re\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport requests\nimport streamlit as st\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nROOT=Path(__file__).resolve().parent\nDATA_PATH=ROOT/"data"/"steam_games_clean.csv"\nVECTORIZER_PATH=ROOT/"artifacts"/"tfidf_vectorizer.joblib"\nMATRIX_PATH=ROOT/"artifacts"/"tfidf_matrix.joblib"\n\nst.set_page_config(page_title="AI Game Assistant",page_icon="🎮",layout="wide")\n\n@st.cache_resource\ndef load_project():\n    if not DATA_PATH.exists():\n        raise FileNotFoundError("data/steam_games_clean.csv is missing. Run the notebook first.")\n    df=pd.read_csv(DATA_PATH).fillna("")\n    vectorizer=joblib.load(VECTORIZER_PATH)\n    matrix=joblib.load(MATRIX_PATH)\n    for col in ["price","positive_ratio","total_reviews","metacritic_score"]:\n        df[col]=pd.to_numeric(df[col],errors="coerce").fillna(0)\n    for col in ["windows","mac","linux"]:\n        df[col]=df[col].astype(str).str.lower().isin(["true","1","yes"])\n    return df,vectorizer,matrix\n\ntry:\n    df,vectorizer,tfidf_matrix=load_project()\nexcept Exception as exc:\n    st.error(str(exc)); st.stop()\n\nPRICE_PATTERN=re.compile(r"(?:under|below|less than|max(?:imum)?|up to)\\s*[£$€]?\\s*(\\d+(?:\\.\\d+)?)",re.I)\n\ndef infer_query_filters(query):\n    q=query.lower()\n    m=PRICE_PATTERN.search(query)\n    max_price=float(m.group(1)) if m else (0.0 if "free" in q else None)\n    platform=None\n    if re.search(r"\\bmac(?:os)?\\b",q): platform="mac"\n    elif re.search(r"\\blinux\\b",q): platform="linux"\n    elif re.search(r"\\bwindows\\b|\\bpc\\b",q): platform="windows"\n    return max_price,platform\n\ndef search_games(query,top_k=5,max_price=None,platform="Any",min_reviews=0):\n    inferred_price,inferred_platform=infer_query_filters(query)\n    if max_price is None: max_price=inferred_price\n    platform=inferred_platform if platform=="Any" else platform.lower()\n    scores=cosine_similarity(vectorizer.transform([query]),tfidf_matrix).ravel()\n    mask=np.ones(len(df),dtype=bool)\n    if max_price is not None: mask &= df["price"].to_numpy(float)<=float(max_price)\n    if platform in {"windows","mac","linux"}: mask &= df[platform].to_numpy(bool)\n    if min_reviews: mask &= df["total_reviews"].to_numpy(float)>=int(min_reviews)\n    valid=np.flatnonzero(mask)\n    if not len(valid): return pd.DataFrame()\n    ranked=valid[np.argsort(scores[valid])[::-1]][:top_k]\n    result=df.iloc[ranked].copy()\n    result.insert(0,"match_score",scores[ranked])\n    return result.reset_index(drop=True)\n\ndef format_price(value):\n    value=float(value)\n    return "Free" if value<=0 else f"£{value:.2f}"\n\ndef fallback_answer(question,results):\n    if results.empty:\n        return "I could not find a game matching those filters. Try widening your requirements."\n    lines=["Here are the closest matches I found:"]\n    for i,row in results.iterrows():\n        ratio=float(row.get("positive_ratio",0)); total=float(row.get("total_reviews",0))\n        reviews=f"{ratio*100:.0f}% positive" if total>0 else "reviews unavailable"\n        lines.append(f\'{i+1}. **{row["name"]}** — {row.get("genres","")} | {format_price(row.get("price",0))} | \'\n                     f\'{row.get("platforms","")} | {reviews}. {row.get("short_description","")}\')\n    return "\\n\\n".join(lines)\n\ndef resolve_groq_model(api_key,preferred=""):\n    r=requests.get("https://api.groq.com/openai/v1/models",headers={"Authorization":f"Bearer {api_key}"},timeout=20)\n    r.raise_for_status()\n    ids=[x.get("id","") for x in r.json().get("data",[])]\n    if preferred and preferred in ids: return preferred\n    for keyword in ["llama","qwen","gemma","mixtral"]:\n        found=next((x for x in ids if keyword in x.lower()),None)\n        if found: return found\n    if not ids: raise RuntimeError("No Groq models available.")\n    return ids[0]\n\ndef build_context(results):\n    return "\\n\\n---\\n\\n".join(\n        f\'Game: {r["name"]}\\nGenres: {r.get("genres","")}\\nPrice: {format_price(r.get("price",0))}\\n\'\n        f\'Platforms: {r.get("platforms","")}\\nPositive ratio: {float(r.get("positive_ratio",0)):.3f}\\n\'\n        f\'Reviews: {int(float(r.get("total_reviews",0)))}\\nDescription: {r.get("short_description","")}\'\n        for _,r in results.iterrows()\n    )\n\ndef answer_with_langchain(question,results,api_key,preferred_model=""):\n    if not api_key:\n        return fallback_answer(question,results),"Retrieval only"\n    try:\n        from langchain_core.prompts import ChatPromptTemplate\n        from langchain_core.runnables import RunnableLambda\n        model=resolve_groq_model(api_key,preferred_model)\n        prompt=ChatPromptTemplate.from_messages([\n            ("system","You are an AI Game Assistant. Use only retrieved context. Do not invent facts."),\n            ("human","Request:\\n{question}\\n\\nRetrieved context:\\n{context}")\n        ])\n        def call_groq(prompt_value):\n            messages=[]\n            for msg in prompt_value.to_messages():\n                role="assistant" if msg.type=="ai" else ("system" if msg.type=="system" else "user")\n                messages.append({"role":role,"content":str(msg.content)})\n            r=requests.post("https://api.groq.com/openai/v1/chat/completions",\n                headers={"Authorization":f"Bearer {api_key}","Content-Type":"application/json"},\n                json={"model":model,"messages":messages,"temperature":0.2},timeout=45)\n            r.raise_for_status()\n            return r.json()["choices"][0]["message"]["content"]\n        text=(prompt | RunnableLambda(call_groq)).invoke({"question":question,"context":build_context(results)})\n        return text,f"LangChain + Groq ({model})"\n    except Exception as exc:\n        return fallback_answer(question,results)+f"\\n\\nLLM unavailable: {type(exc).__name__}","Retrieval fallback"\n\nst.title("AI Game Assistant")\nst.caption("Steam game discovery with local retrieval, natural-language filters, LangChain and an optional Groq LLM.")\n\nwith st.sidebar:\n    st.header("Settings")\n    api_key=st.text_input("Groq API key (optional)",type="password",value=os.getenv("GROQ_API_KEY",""))\n    preferred_model=st.text_input("Preferred Groq model (optional)",value=os.getenv("GROQ_MODEL",""))\n    top_k=st.slider("Number of games",3,10,5)\n    platform=st.selectbox("Platform",["Any","Windows","Mac","Linux"])\n    use_price=st.checkbox("Set maximum price")\n    max_price=st.slider("Maximum price (£)",0,100,30) if use_price else None\n    min_reviews=st.number_input("Minimum review count",min_value=0,value=0,step=100)\n    st.caption(f"Dataset contains {len(df):,} games.")\n\nif "messages" not in st.session_state: st.session_state.messages=[]\nfor msg in st.session_state.messages:\n    with st.chat_message(msg["role"]): st.markdown(msg["content"])\n\nquery=st.chat_input(\'Ask for a game, e.g. "co-op horror under £20 for Windows"\')\nif query:\n    st.session_state.messages.append({"role":"user","content":query})\n    with st.chat_message("user"): st.markdown(query)\n    results=search_games(query,top_k,max_price,platform,min_reviews)\n    answer,mode=answer_with_langchain(query,results,api_key,preferred_model)\n    with st.chat_message("assistant"):\n        st.markdown(answer); st.caption(f"Mode: {mode}")\n        if not results.empty:\n            st.subheader("Retrieved games")\n            for _,row in results.iterrows():\n                c1,c2=st.columns([1,3])\n                image=str(row.get("header_image","")).strip()\n                with c1:\n                    if image.startswith("http"): st.image(image,use_container_width=True)\n                with c2:\n                    st.markdown(f\'### {row["name"]}\')\n                    st.write(row.get("short_description",""))\n                    st.write(f\'**Genres:** {row.get("genres","Not listed")}  |  **Price:** {format_price(row.get("price",0))}  |  **Platforms:** {row.get("platforms","Not listed")}\')\n                    if float(row.get("total_reviews",0))>0:\n                        st.write(f\'**Positive reviews:** {float(row.get("positive_ratio",0))*100:.1f}%\')\n                st.divider()\n    st.session_state.messages.append({"role":"assistant","content":answer})\n'
APP_PATH = PROJECT_ROOT / "app.py"
APP_PATH.write_text(APP_CODE, encoding="utf-8")
print("Created:", APP_PATH)

Created: /Users/adityaraj/Desktop/AI/app.py


## 14. Create project files

In [14]:
REQUIREMENTS = 'pandas>=2.2\nnumpy>=1.26\nscikit-learn>=1.5\nrequests>=2.31\njoblib>=1.3\nstreamlit>=1.35\nlangchain-core>=0.3\n'
README = '# AI Game Assistant\n\nPortfolio-ready game discovery assistant built with Python, Streamlit and LangChain.\n\n## Features\n- Steam game data ingestion and cleaning\n- TF-IDF retrieval over titles, genres, tags, categories and descriptions\n- natural-language price and platform filters\n- optional LangChain + Groq conversational answers\n- retrieval-only fallback with no API key\n- Streamlit chat UI\n\n## Run\n\n```bash\npython -m pip install -r requirements.txt\nstreamlit run app.py\n```\n\nThe notebook creates `data/steam_games_clean.csv` and the retrieval files in `artifacts/`.\n\n## Data\nPrimary source: `FronkonGames/steam-games-dataset` on Hugging Face. The notebook reuses a local cleaned dataset first, then tries the Dataset Viewer API, then falls back to bundled demo data if necessary.\n'
GITIGNORE = '.DS_Store\n.ipynb_checkpoints/\n__pycache__/\n*.pyc\n.env\n.venv/\nvenv/\n'

(PROJECT_ROOT/"requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
(PROJECT_ROOT/"README.md").write_text(README, encoding="utf-8")
(PROJECT_ROOT/".gitignore").write_text(GITIGNORE, encoding="utf-8")
print("Created requirements.txt, README.md and .gitignore")

Created requirements.txt, README.md and .gitignore


## 15. Validate everything

In [15]:
import py_compile
py_compile.compile(str(PROJECT_ROOT/"app.py"), doraise=True)

required_files=[
    CLEAN_DATA_PATH,VECTORIZER_PATH,MATRIX_PATH,
    PROJECT_ROOT/"app.py",PROJECT_ROOT/"requirements.txt",PROJECT_ROOT/"README.md",PROJECT_ROOT/".gitignore"
]
for path in required_files:
    assert path.exists(), f"Missing: {path}"

assert len(df)>=10
assert tfidf_matrix.shape[0]==len(df)
assert len(vectorizer.vocabulary_)>10
assert not search_games("fantasy RPG",3).empty

print("All smoke tests passed.")
for path in required_files:
    print(" -", path.relative_to(PROJECT_ROOT))

All smoke tests passed.
 - data/steam_games_clean.csv
 - artifacts/tfidf_vectorizer.joblib
 - artifacts/tfidf_matrix.joblib
 - app.py
 - requirements.txt
 - README.md
 - .gitignore


## 16. Launch command

In [16]:
print("From this project folder, run:")
print(f'"{sys.executable}" -m pip install -r requirements.txt')
print(f'"{sys.executable}" -m streamlit run app.py')
print("\nGroq API key is optional; without it the app still returns retrieved recommendations.")

From this project folder, run:
"/Library/Frameworks/Python.framework/Versions/3.13/bin/python3" -m pip install -r requirements.txt
"/Library/Frameworks/Python.framework/Versions/3.13/bin/python3" -m streamlit run app.py

Groq API key is optional; without it the app still returns retrieved recommendations.


## Architecture

```text
Steam data
  ↓
cleaning + review features
  ↓
TF-IDF local retrieval
  ↓
natural-language price/platform filters
  ↓
top relevant games
  ↓
LangChain prompt pipeline (optional Groq LLM)
  ↓
Streamlit chat interface
```

The core project does **not** import `transformers`, `sentence-transformers`, `PEFT`, `huggingface_hub`, or `FAISS`, which removes the dependency chain that caused the earlier `HybridCache` and `as_extended_path` errors.